## Reliable Agentic Workflow

In [2]:
# Query -> Router -> Tool -> Final
# Query -> Router -> Validation (If failed, retry) -> Tool (If no tool then Fallback) -> Final

In [3]:
# Scenario 1:

# q='what is twenty multiplied by three?'
# cq='25*three'
# validation failed then clean the query again and retry validation

# Scenario 2:

# T1- Maths
# T2- Time
# Query= 'What is AI?'
# Router-> No tool selection then use the fallback option

In [9]:
import os
import re
from typing import TypedDict
from datetime import datetime
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

### LLM

In [5]:
llm = ChatOpenAI(
    model='gpt-4o-mini',
    api_key=os.getenv('OPENAI_SECRET_KEY'),
    temperature=0
)

llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000027CB9E8E690>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000027CBA34CC50>, root_client=<openai.OpenAI object at 0x0000027CB9E635D0>, root_async_client=<openai.AsyncOpenAI object at 0x0000027CBA877090>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

### State

In [8]:
class AgentState(TypedDict):
    query: str
    tool: str
    tool_input: str
    result: str
    
AgentState

__main__.AgentState

### Tools

In [27]:
def calculator_tool(expression: str) -> str:
    print('\n[Calculator Tool]: ', expression)
    return eval(expression)

def current_time_tool(_:str = '') -> str:
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

### Nodes

In [20]:
# Router Node

def router_node(state):
    print('\n[ROUTER NODE]')
    query = state['query']
    prompt = f"""Decide which tool to be used for the given query:
Tools:
    - calculator
    - current_time
    - fallback
Query:
{query}

Return only the tool name"""

    result = llm.invoke(prompt).content.strip().lower()

    return {'tool':result}

router_node

<function __main__.router_node(state)>

In [21]:
# Math Node

def clean_math_node(state):
    print('\n[CLEAN MATH NODE]')
    text = state['query']
    text = text.replace(',', '').replace('\\', '')
    text = text.replace('x', '*')
    
    match = re.findall(r'[0-9+\-/*%^().]+', text)
    
    if match:
        return {'tool_input': match[0].strip()}
    
    return {'tool_input': text.strip()}

In [23]:
# Validation Node

def validation_node(state):
    print('\n[VALIDATION NODE]')
    
    expression = state['tool_input']
    
    try:
        eval(expression)
        print('Validation Passes')
        return state
    except:
        print('Validation Failed')
        return {'tool': 'retry'}

In [33]:
# Retry Node

def retry_node(state):
    print('\n[RETRY NODE]')
    
    query = state['query']
    
    retry_prompt = f'''Strictly return the valid math expression from the given query.
Do not return any extra text or explanation.
Example: 25*3

QUERY:
{query}
'''

    response = llm.invoke(retry_prompt).content.strip()
    return {'tool_input': response}
    
retry_node

<function __main__.retry_node(state)>

In [28]:
def calculator_node(state):
    print('\n[CALCULATOR NODE]')
    result = calculator_tool(state['tool_input'])
    return {'result': result}

In [30]:
def current_time_node(state):
    print('\n[CURRENT TIME NODE]')
    result = current_time_tool(state['tool_input'])
    
    return {'result': result}

In [36]:
def fallback_node(state):
    print('\n[FALLBACK NODE]')
    response = llm.invoke(state['query'])
    
    return {'result': response}